# Lab 1b · Agent Framework y un agente que programa

El mismo agente del Lab 1a… en 10 líneas. Después le darás un **puesto de trabajo** y una **herramienta
propia**, construirás un **agente que escribe y ejecuta código** (como los agentes de código profesionales),
pedirás **salida estructurada** y lo verás por dentro en **DevUI**.

⏱️ 24 minutos. Camino mínimo: pasos **1, 2 y 3**. Si vas rápido: pasos 4-5 y ⭐.

In [ ]:
import json
from typing import Annotated, Literal

import pandas as pd
from agent_framework import Agent, tool
from pydantic import BaseModel, Field

import mucdia
from mucdia import negocio

cliente = mucdia.cliente()  # cliente de Gemini con límite de llamadas, reintentos, contador y telemetría
mucdia.reiniciar_traza()

## Paso 1 · El mismo agente en 10 líneas

In [ ]:
agente = Agent(
    client=cliente,
    name="analista",
    instructions=(
        "Eres el analista de datos de MUCDIA (cadena ficticia). Usa las herramientas y no inventes datos. "
        "Hay datos del 2025-01-01 al 2026-10-31. Responde en español, breve y con cifras."
    ),
    tools=[negocio.ventas_por_categoria, negocio.ventas_por_tienda, negocio.top_productos],
)

respuesta = await agente.run(
    "¿Qué categoría vendió más en el tercer trimestre de 2026 y qué tienda vendió más de esa categoría? "
    "Compáralo con el tercer trimestre de 2025."
)
print(respuesta.text)
mucdia.mostrar_traza(respuesta)  # los pasos que ha dado el agente por dentro

**¿Qué ha hecho el framework por ti?** El bucle, la ejecución de herramientas (también en paralelo), los
errores, las firmas de razonamiento… y el **JSON Schema**, igual que tu función `esquema()` del lab 1a.
También te da la **telemetría** de cada llamada (tokens, milisegundos y herramientas pedidas):

In [ ]:
print(json.dumps(tool(negocio.top_productos).to_json_schema_spec(), indent=2, ensure_ascii=False))
mucdia.traza_df()

## Paso 2 · ✏️ El puesto de trabajo y tu herramienta

Las **instrucciones** son la descripción del puesto: rol, qué hacer siempre, qué no hacer nunca y el formato.
La **herramienta** es una función con *type hints*, descripciones en `Annotated` y un **docstring** claro.
Escribe las de vuestra área. Mira las tablas con `print(mucdia.esquema())`. Ideas:

| Equipo | Herramienta |
|---|---|
| Finanzas | margen bruto por categoría o por tienda |
| Marketing | clientes por segmento, ventas por canal (tienda/web/app) |
| Operaciones y Clientes | tickets por categoría y tienda, tiempos de resolución |
| Documentación y Compliance | clientes con/sin consentimiento de marketing, reseñas por producto |

In [ ]:
INSTRUCCIONES_EQUIPO = """✏️ TODO: describe el puesto del agente de tu equipo:
- ¿Quién es y qué le preocupa?
- ¿Qué debe hacer siempre (verificar con datos, citar cifras...)? ¿Qué no debe hacer nunca?
- ¿Qué formato y longitud de respuesta esperas?"""


@tool
def mi_herramienta(
    desde: Annotated[str, "Fecha inicial AAAA-MM-DD"],
    hasta: Annotated[str, "Fecha final AAAA-MM-DD"],
) -> str:
    """✏️ TODO: describe qué devuelve. El modelo decide usarla leyendo SOLO esto."""
    df = mucdia.consultar(
        "SELECT ... FROM v_ventas_lineas WHERE estado = 'completada' AND fecha BETWEEN ? AND ?",  # ✏️ TODO
        (desde, hasta),
    )
    return df.to_csv(index=False)


MI_HERRAMIENTA = mi_herramienta
print(MI_HERRAMIENTA.func("2026-07-01", "2026-09-30"))  # pruébala antes de dársela al agente

In [ ]:
agente_equipo = Agent(
    client=cliente,
    name="agente_equipo",
    instructions=INSTRUCCIONES_EQUIPO,
    tools=[MI_HERRAMIENTA, negocio.ventas_por_categoria, negocio.ventas_por_tienda, negocio.objetivos_vs_real],
)
respuesta = await agente_equipo.run("✏️ TODO: una pregunta de tu área que obligue a usar tu herramienta")
print(respuesta.text)
mucdia.mostrar_traza(respuesta)

## Paso 3 · ✏️ Un agente que escribe código

Un agente puede actuar de dos formas: llamando a **herramientas fijas** (JSON, como hasta ahora) o
**escribiendo y ejecutando código** (*CodeAct*, lo que hacen los agentes de código). Hay preguntas que tus
herramientas fijas no saben responder:

In [ ]:
pregunta_dificil = ("¿Qué porcentaje de los clientes que compraron en el primer semestre de 2025 volvió a comprar "
                    "en el primer semestre de 2026? ¿Cambia según el canal de su primera compra (tienda, web, app)?")
mucdia.reiniciar_traza()   # medimos desde aquí: herramientas fijas frente a código
print((await agente.run(pregunta_dificil)).text)

Dale al agente una herramienta que ejecuta Python. `mucdia.ejecutar_codigo()` lo ejecuta en un **proceso
aparte**, con un **límite de tiempo**, **sin tus claves** en el entorno, la base de datos en **solo lectura**
y la salida **recortada** (para no llenar el contexto). ✏️ Tu trabajo es diseñar la **interfaz**: el docstring
(lo único que ve el modelo de la herramienta) y las instrucciones del agente.

In [ ]:
@tool
def ejecutar_python(codigo: Annotated[str, "Código Python que calcula la respuesta y la imprime con print()"]) -> str:
    """✏️ TODO: describe la herramienta para el modelo:
    - qué tiene disponible el código (pd, np y consultar(sql), que devuelve un DataFrame de la base de datos de MUCDIA)
    - qué tabla o vista le conviene usar para las ventas (mira print(mucdia.esquema()))
    - que imprima solo el resultado final (la salida se recorta) y que hay un límite de tiempo"""
    return mucdia.ejecutar_codigo(codigo, segundos=20, max_caracteres=3000)


INSTRUCCIONES_CODIGO = """✏️ TODO: instrucciones del analista que programa:
- ¿Cómo debe trabajar (calcular con la herramienta, corregir sus errores)?
- ¿Qué datos hay y qué filtro no puede olvidar?
- ¿Qué debe comprobar antes de responder? ¿Qué formato quieres?"""

In [ ]:
analista_programador = Agent(client=cliente, name="analista_programador", instructions=INSTRUCCIONES_CODIGO,
                             tools=[ejecutar_python])
respuesta_codigo = await analista_programador.run(pregunta_dificil)
print(respuesta_codigo.text)
mucdia.mostrar_traza(respuesta_codigo, n=500)

🤔 Para pensar (y comentar con tu equipo):
- ¿Qué código ha escrito? La descripción de la herramienta pedía usar `estado = 'completada'`: ¿lo ha hecho?
  ¿Coincide lo que **dice** su respuesta con lo que **hace** su código? ¿Llevarías la cifra a un comité?
- ¿Ha fallado con `NameError`? Cada ejecución es un **proceso nuevo**: las variables de la anterior ya no
  existen. ¿Cómo se lo explicarías al modelo?
- Mira `mucdia.traza_df()` (columna `herramientas`): ¿cuántas llamadas y cuántos tokens ha necesitado el que
  programa frente al de herramientas fijas?
- **¿Y si la pregunta la escribiera un cliente en un chat?** Un agente que ejecuta código tiene muchísimo
  poder: podría leer ficheros, conectarse a internet… En la sesión 3 veremos cómo se ataca un agente.

In [ ]:
mucdia.traza_df()

## Paso 4 · Salida estructurada

Si la respuesta la va a consumir otro sistema (un Excel, un cuadro de mando, otro agente…), pide un **esquema**.
El modelo genera JSON que cumple el esquema (decodificación restringida) y Pydantic lo valida.

In [ ]:
class CategoriaTrimestre(BaseModel):
    categoria: str
    ventas_eur: float
    variacion_pct: float = Field(description="Variación porcentual frente al mismo trimestre del año anterior")


class InformeTrimestral(BaseModel):
    trimestre: str
    categorias: list[CategoriaTrimestre]
    conclusion: str = Field(description="Una frase para la dirección")
    confianza: Literal["alta", "media", "baja"] = Field(description="Confianza en la conclusión según los datos usados")


r = await agente.run("Prepara el informe del T3 de 2026 por categoría comparado con el T3 de 2025.",
                     options={"response_format": InformeTrimestral})
informe = r.value
print(informe.conclusion, f"(confianza {informe.confianza})")
pd.DataFrame([c.model_dump() for c in informe.categorias])

## Paso 5 · Memoria y DevUI

Por defecto cada `run` empieza de cero. Con una **sesión**, el agente recuerda lo hablado (reenviando el
historial: más tokens en cada llamada).

In [ ]:
sesion = agente.create_session()
r1 = await agente.run("¿Cuánto vendió la tienda de Valencia (T04) en el primer trimestre de 2026?", session=sesion)
print(r1.text, "\n")
r2 = await agente.run("¿Y en el mismo trimestre de 2025? ¿A qué crees que se debe la diferencia?", session=sesion)
print(r2.text)

🤔 El agente solo ve datos de ventas: para explicar el *porqué* puede que especule. ¿Dónde estaría la respuesta?
(Pista: en los documentos internos de MUCDIA… lo resolveremos en la **sesión 2** con RAG.)

**DevUI** te enseña lo mismo en una interfaz web:
1. Abre un **terminal** (☰ → Terminal → New Terminal) y ejecuta `python sesion1/app_devui.py`.
2. Abre el puerto **8080** (pestaña **PUERTOS** → 🌐).
3. Pregunta: *"Las ventas de Hogar en Andalucía han caído en el T3 de 2026. ¿Qué ha pasado?"* y mira la traza.

Para parar el servidor: `Ctrl + C` en el terminal.

## ⭐ Para ir más allá: el agente que programa, ¿valida bien?

Pídele al analista programador un modelo predictivo y revisa **cómo lo valida**. ¿Separa entrenamiento y
prueba respetando el tiempo, o mezcla el futuro con el pasado (fuga de datos)? ¿Lo compara con un modelo
ingenuo? ¿Cómo ha construido las semanas (¿de lunes a domingo?, ¿hay semanas incompletas)?

In [ ]:
r = await analista_programador.run(
    "Ajusta una regresión lineal que prevea las ventas semanales de Electrónica a partir del descuento medio de "
    "la semana y de las ventas de la semana anterior. Dime su error en un periodo de prueba y cómo has validado.")
print(r.text)
mucdia.mostrar_traza(r, n=500)

## 🧩 Reto de equipo: Agent Canvas

Para casa: el **Agent Canvas** ([docs/agent_canvas.md](../docs/agent_canvas.md)) del agente de vuestra área
(problema, éxito, herramientas, autonomía, riesgos, supervisión, coste y valor, regulación).
En la sesión 2 conectaréis ese agente a los datos, a vuestros modelos y a los documentos de MUCDIA.

In [ ]:
mucdia.uso()